# Treino do wake word **"Ei Jarvis"** (PT-BR) — NexusIDE

Runtime → Change runtime type → **T4 GPU** → Run all (~40 min).
Ao final, baixe `ei_jarvis.onnx` e copie para `~/.nexus/models/wake/custom.onnx`.

In [ ]:
#@title 1) Dependências (openWakeWord treino + edge-tts PT-BR)
!pip -q install openwakeword==0.6.0 edge-tts==6.1.12 onnxruntime==1.19.2 torch torchaudio audiomentations==0.35.0 torch-audiomentations==0.11.0 speechbrain==0.5.16 mutagen acoustics pronouncing datasets deep-phonemizer
import openwakeword, os
openwakeword.utils.download_models()  # mel + embedding oficiais
print('ok')

In [ ]:
#@title 2) Gerar ~440 positivos "ei jarvis" com vozes neurais PT-BR
import asyncio, edge_tts, os, itertools, random
os.makedirs('positives_raw', exist_ok=True)
VOICES = ['pt-BR-AntonioNeural','pt-BR-FranciscaNeural','pt-BR-ThalitaNeural',
          'pt-PT-DuarteNeural','pt-PT-RaquelNeural']
TEXTS  = ['ei jarvis','Ei, Jarvis!','ei járvis','Ei Jarvis?','ei jarvis.']
RATES  = ['-20%','-10%','+0%','+10%','+25%']
PITCH  = ['-20Hz','-10Hz','+0Hz','+10Hz']
async def gen():
    combos = list(itertools.product(VOICES, TEXTS, RATES, PITCH))
    random.shuffle(combos)
    n = 0
    for v,t,r,p in combos[:440]:
        f = f'positives_raw/{n:04d}.mp3'
        try:
            await edge_tts.Communicate(t, v, rate=r, pitch=p).save(f); n += 1
        except Exception as e:
            print('skip', e)
    print('gerados:', n)
await gen()
# mp3 -> wav 16k mono
!mkdir -p positives && for f in positives_raw/*.mp3; do ffmpeg -v error -y -i "$f" -ar 16000 -ac 1 "positives/$(basename ${f%.mp3}).wav"; done
!ls positives | wc -l

In [ ]:
#@title 3) Augmentação (ruído + reverb) → mais robustez
import glob, numpy as np, scipy.io.wavfile as wavfile, os
from audiomentations import Compose, AddColorNoise, RoomSimulator, Gain, PitchShift
os.makedirs('positives_aug', exist_ok=True)
aug = Compose([AddColorNoise(p=0.7), Gain(min_gain_db=-8, max_gain_db=6, p=0.8),
               PitchShift(min_semitones=-2, max_semitones=2, p=0.4)])
i = 0
for f in glob.glob('positives/*.wav'):
    sr, x = wavfile.read(f)
    x = x.astype(np.float32)/32768
    for k in range(3):
        y = aug(samples=x, sample_rate=16000)
        pad = np.zeros(16000, dtype=np.float32)  # 1s de silêncio em volta
        y2 = np.concatenate([pad, y, pad])
        wavfile.write(f'positives_aug/{i:05d}.wav', 16000, (np.clip(y2,-1,1)*32767).astype(np.int16)); i+=1
print('aumentados:', i)

In [ ]:
#@title 4) Features dos positivos + negativos oficiais pré-computados
import numpy as np, glob, scipy.io.wavfile as wavfile
from openwakeword.utils import AudioFeatures
F = AudioFeatures()
pos = []
for f in sorted(glob.glob('positives_aug/*.wav')):
    sr, x = wavfile.read(f)
    feats = F.embed_clips([x.astype(np.int16)], batch_size=16)[0]  # [n,96] por janela
    # janelas deslizantes de 16 frames com a frase presente
    for s in range(0, max(1, feats.shape[0]-16), 4):
        w = feats[s:s+16]
        if w.shape[0]==16: pos.append(w)
pos = np.array(pos, dtype=np.float32)
print('janelas positivas:', pos.shape)
# negativos oficiais (ACAV100M features ~2GB)
from huggingface_hub import hf_hub_download
neg_path = hf_hub_download(repo_id='davidscripka/openwakeword_features',
    filename='openwakeword_features_ACAV100M_2000_hrs_16bit.npy', repo_type='dataset')
neg = np.load(neg_path, mmap_mode='r')
print('negativos:', neg.shape)

In [ ]:
#@title 5) Treinar a cabeça (16×96 → score) e exportar ONNX
import torch, torch.nn as nn, numpy as np
dev = 'cuda' if torch.cuda.is_available() else 'cpu'
class Head(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(nn.Flatten(), nn.Linear(16*96, 128), nn.LayerNorm(128),
                                 nn.ReLU(), nn.Linear(128, 1), nn.Sigmoid())
    def forward(self, x): return self.net(x)
m = Head().to(dev)
opt = torch.optim.AdamW(m.parameters(), lr=1e-3)
lossf = nn.BCELoss()
posT = torch.tensor(pos, device=dev)
STEPS, B = 6000, 512
for step in range(STEPS):
    ni = np.random.randint(0, neg.shape[0]-16, size=B)
    nb = np.stack([neg[j:j+16] for j in ni]).astype(np.float32)
    xb = torch.cat([posT[torch.randint(0, posT.shape[0], (B//4,))],
                    torch.tensor(nb, device=dev)])
    yb = torch.cat([torch.ones(B//4,1, device=dev), torch.zeros(B,1, device=dev)])
    opt.zero_grad(); loss = lossf(m(xb), yb); loss.backward(); opt.step()
    if step % 500 == 0: print(step, float(loss))
m.eval()
torch.onnx.export(m.cpu(), torch.zeros(1,16,96), 'ei_jarvis.onnx',
                  input_names=['x.1'], output_names=['y'], opset_version=17)
print('exportado: ei_jarvis.onnx')

In [ ]:
#@title 6) Validar + baixar
import onnxruntime as rt, numpy as np
s = rt.InferenceSession('ei_jarvis.onnx')
print('sanidade (pos deve > neg):',
      float(s.run(None, {'x.1': pos[:1]})[0]),
      float(s.run(None, {'x.1': np.zeros((1,16,96), np.float32)})[0]))
from google.colab import files
files.download('ei_jarvis.onnx')
print('Instale: cp ei_jarvis.onnx ~/.nexus/models/wake/custom.onnx e reinicie o app')